In [ ]:
from agent import Categorizer, CategorizerConfig


# Initialize categorizer with categories
categories = [
  "Finance Folder",
  "Operations Folder",
  "Marketing Folder",
  "Marketing/Competitive Intelligence Folder",
  "Marketing/Advertising Folder",
  "Marketing/Brand Folder",
  "Operations/Supply Chain Folder",
  "Operations/Invoices Folder"
]

# Basic usage
categorizer = Categorizer(categories)
result = categorizer("Proposal for Equity Investment")

# With custom configuration
config = CategorizerConfig(
  threshold=0.2,
  top_k=3
)
categorizer = Categorizer(categories, config=config)

# Get top categories with scores
results = categorizer(
  "New Marketing Campaign",
  return_scores=True,
  top_k=2
)

print(results)

# Batch processing
texts = [
  "Investment Proposal",
  "Marketing Strategy",
  "Supply Chain Update"
]
results = categorizer(texts)

print(results)


# Update categories
categorizer.add_categories(["HR Folder", "Law Folder"])



In [4]:
from agent import Summarizer, SummarizerConfig

# Initialize with default configuration
summarizer = Summarizer()

# Or initialize with custom configuration
config = SummarizerConfig(
  max_length=150,
  num_beams=6,
  summary_length="brief",
  temperature=0.8,
  do_sample=True
)
summarizer = Summarizer(config=config)

# Update configuration dynamically
summarizer.update_config(
  num_beams=8,
  temperature=0.7
)

# Generate summaries
text = """Videos that say approved vaccines are dangerous and cause autism, cancer or infertility are among those that will be taken down, the company said.  The policy includes the termination of accounts of anti-vaccine influencers.  Tech giants have been criticised for not doing more to counter false health information on their sites.  In July, US President Joe Biden said social media platforms were largely responsible for people's scepticism in getting vaccinated by spreading misinformation, and appealed for them to address the issue.  YouTube, which is owned by Google, said 130,000 videos were removed from its platform since last year, when it implemented a ban on content spreading misinformation about Covid vaccines.  In a blog post, the company said it had seen false claims about Covid jabs "spill over into misinformation about vaccines in general". The new policy covers long-approved vaccines, such as those against measles or hepatitis B.  "We're expanding our medical misinformation policies on YouTube with new guidelines on currently administered vaccines that are approved and confirmed to be safe and effective by local health authorities and the WHO," the post said, referring to the World Health Organization."""
summary = summarizer(text)

# Generate with custom one-time generation config
from transformers import GenerationConfig
custom_config = GenerationConfig(
  max_length=200,
  num_beams=5,
  temperature=0.9
)
summary = summarizer(text, generation_config=custom_config)

summary

'YouTube has updated its medical misinformation policy on Covid vaccines.'

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

def compare_embedding_with_sentence(sentence, embeddings, model: SentenceTransformer):
  # Encode the new sentence
  new_embedding = model.encode([sentence])
  
  # Calculate cosine similarity
  similarities = cosine_similarity(new_embedding, embeddings)
  
  return similarities

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

def compare_embedding_with_sentence(sentence, embeddings, model: SentenceTransformer):
  # Encode the new sentence
  new_embedding = model.encode([sentence])
  
  # Calculate cosine similarity
  similarities = cosine_similarity(new_embedding, embeddings)
  
  return similarities

# Example usage
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

categories = ["Finance Folder", "Operations Folder", "Marketing Folder", "Marketing/Competitive Intelligence Folder", "Marketing/Advertising Folder", "Marketing/Brand Folder", "Operations/Supply Chain Folder",  "Operations/Invoices Folder"]

# Sentences are encoded by calling model.encode()
category_embeddings = model.encode(categories)

sentence_to_compare = "Proposal for Equity Investment in Advance Chemicals"
similarities = compare_embedding_with_sentence(sentence_to_compare, category_embeddings, model)

print(similarities) # [[0.22507085 0.14186437 0.13375607 0.19272968 0.12795648 0.1362828 0.18207154 0.1512896 ]]

In [ ]:
#!pip install transformers[sentencepiece]
from transformers import pipeline
classifier = pipeline("zero-shot-classification", model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli")



In [ ]:
sequence_to_classify = "Angela Merkel is a politician in Germany and leader of the CDU"
candidate_labels = ["politics", "economy", "entertainment", "environment"]
output = classifier(sequence_to_classify, candidate_labels, multi_label=False)
print(output)

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
from transformers import AutoModel, AutoTokenizer, AutoModelForSeq2SeqLM

device = 'cpu' #or 'cpu' for translate on cpu

model_name = 'utrobinmv/t5_summary_en_ru_zh_base_2048'
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.eval()
model.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

text = """Videos that say approved vaccines are dangerous and cause autism, cancer or infertility are among those that will be taken down, the company said.  The policy includes the termination of accounts of anti-vaccine influencers.  Tech giants have been criticised for not doing more to counter false health information on their sites.  In July, US President Joe Biden said social media platforms were largely responsible for people's scepticism in getting vaccinated by spreading misinformation, and appealed for them to address the issue.  YouTube, which is owned by Google, said 130,000 videos were removed from its platform since last year, when it implemented a ban on content spreading misinformation about Covid vaccines.  In a blog post, the company said it had seen false claims about Covid jabs "spill over into misinformation about vaccines in general". The new policy covers long-approved vaccines, such as those against measles or hepatitis B.  "We're expanding our medical misinformation policies on YouTube with new guidelines on currently administered vaccines that are approved and confirmed to be safe and effective by local health authorities and the WHO," the post said, referring to the World Health Organization."""

# text summary generate
prefix = 'summary: '
src_text = prefix + text
input_ids = tokenizer(src_text, return_tensors="pt")

generated_tokens = model.generate(**input_ids.to(device))

result = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
print(result)
#YouTube is cracking down on videos that suggest Covid-19 vaccines are dangerous and harmful.

# text brief summary generate
prefix = 'summary brief: '
src_text = prefix + text
input_ids = tokenizer(src_text, return_tensors="pt")

generated_tokens = model.generate(**input_ids.to(device))

result = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
print(result)
#YouTube is cracking down on misleading information about Covid vaccines.

# text big summary generate
prefix = 'summary big: '
src_text = prefix + text
input_ids = tokenizer(src_text, return_tensors="pt")

generated_tokens = model.generate(**input_ids.to(device))

result = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
print(result)
#YouTube has said it will remove more than 1,500 videos of Covid vaccines from its platform in a bid to tackle the spread of misinformation about the jabs.
